# 07 · Pseudo-Label Generation v2 — GradCAM++ Multi-Scale

Cải tiến pseudo-label so với notebook 04 dựa trên 5 vấn đề đã phân tích:
1. **Finer target layer** (`features.6`, 14×14 thay vì 7×7) → mask sắc nét hơn
2. **GradCAM++** thay GradCAM — second-order gradients, tốt hơn cho multiple small spots
3. **Multi-scale fusion** (0.6 × fine + 0.4 × coarse) — kết hợp detail + context
4. **Adaptive Otsu threshold** thay fixed=0.5 — tự điều chỉnh theo distribution từng ảnh
5. **Connected component filtering** — loại noise nhỏ <50px, giữ từng đốm riêng lẻ

**Output:** Masks tại `data/pseudo_labels_v2_train/` · Ảnh tại `data/processed_v2_train/` · Figures tại `visualizations/pseudo_labeling_v2/`

In [ ]:
import os
import sys
import json
proj_root = r'e:\Master\thesis-clean'
sys.path.insert(0, proj_root)
sys.path.insert(0, os.path.join(proj_root, 'utils'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import cv2
from tqdm import tqdm
from pathlib import Path
from PIL import Image
from collections import defaultdict
import torch
import torch.nn as nn
from torchvision import transforms

from utils.models import EfficientNetClassifier

# Plot style — nhất quán với notebook 01-04
plt.style.use('default')
sns.set_palette("husl")

# Class metadata — nhất quán với notebook 01-04
CLASS_NAMES  = ["ALGAL_LEAF_SPOT", "ALLOCARIDARA_ATTACK", "HEALTHY_LEAF", "LEAF_BLIGHT", "PHOMOPSIS_LEAF_SPOT"]
CLASS_LABELS = ["Algal Leaf Spot", "Allocaridara Attack", "Healthy Leaf", "Leaf Blight", "Phomopsis Leaf Spot"]
CLASS_COLORS = ["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IDX_TO_LABEL = {i: label for i, label in enumerate(CLASS_LABELS)}

VIZ_DIR = "visualizations/pseudo_labeling_v2"
os.makedirs(VIZ_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"proj_root: {proj_root}")

## Load Data + Model

Cùng 3,104 ảnh train với notebook 03-04. Class index gán theo thứ tự alphabet của CLASS_NAMES — nhất quán với `ImageFolder` trong notebook 02.

Load `best_model.pth` — cùng checkpoint với notebook 03-04 để đảm bảo pseudo-label v2 sinh ra từ cùng một model, có thể so sánh công bằng với v1.

In [ ]:
train_dir = os.path.join(proj_root, 'notebooks', 'data', 'raw', 'train')
print(f"Train dir: {train_dir}")

selected_samples = []
for cls_name in CLASS_NAMES:
    cls_dir = Path(train_dir) / cls_name
    cls_idx = CLASS_TO_IDX[cls_name]
    imgs = sorted(list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")))
    for img_path in imgs:
        selected_samples.append((str(img_path), cls_idx, cls_name))
    print(f"  {cls_name}: {len(imgs)} ảnh")

print(f"\nTổng: {len(selected_samples)} ảnh")

In [ ]:
model = EfficientNetClassifier(num_classes=len(CLASS_NAMES), backbone='efficientnet_b0', pretrained=False)

ckpt_path = 'models/classification/checkpoints/best_model.pth'
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Không tìm thấy {ckpt_path}. Chạy notebook 02 trước.")

checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()
print(f"Loaded: {ckpt_path} — val_acc at save: {checkpoint.get('val_acc', 'N/A'):.4f}")

## GradCAM++ Multi-Scale Setup

Hai target layer thay vì một:
- **`features.6` (14×14)**: spatial resolution cao hơn → bắt được các đốm nhỏ mà 7×7 không đủ chi tiết
- **`features.8` (7×7)**: semantic context, biết "đây là vùng bệnh" nhưng không biết chính xác pixel nào

Fusion theo weighted sum (0.6×fine + 0.4×coarse): ưu tiên fine map để mask sắc nét hơn, nhưng vẫn giữ context từ coarse layer.

GradCAM++ dùng second-order gradient approximation thay vì chỉ average gradient của GradCAM — tốt hơn khi có nhiều instance của cùng class trong một ảnh (nhiều đốm rời).

In [6]:
def get_last_conv_in_block(model, block_idx):
    """Lấy tên Conv2d cuối cùng trong features block chỉ định."""
    prefix = f'base.features.{block_idx}'
    candidates = []
    for name, module in model.named_modules():
        if name.startswith(prefix) and isinstance(module, nn.Conv2d):
            candidates.append(name)
    if not candidates:
        raise ValueError(f'Không tìm thấy Conv2d trong {prefix}')
    return candidates[-1]

layer6_name = get_last_conv_in_block(model, 6)  # 14×14 resolution
layer8_name = get_last_conv_in_block(model, 8)  # 7×7  resolution

print('Layer fine  (features.6):', layer6_name)
print('Layer coarse(features.8):', layer8_name)

Layer fine  (features.6): base.features.6.3.block.3.0
Layer coarse(features.8): base.features.8.0


In [7]:
class MultiScaleGradCAMpp:
    """
    GradCAM++ trên 2 layer (fine 14×14 + coarse 7×7), fuse bằng weighted sum.
    GradCAM++ weights dùng second-order gradient approximation (tốt hơn GradCAM
    cho bệnh có nhiều đốm nhỏ vì không bị average-out).
    """

    def __init__(self, model, layer_fine_name, layer_coarse_name, device, weight_fine=0.6):
        self.model = model.to(device)
        self.model.eval()
        self.device = device
        self.weight_fine   = weight_fine
        self.weight_coarse = 1.0 - weight_fine

        modules = dict(model.named_modules())
        self.layer_fine   = modules[layer_fine_name]
        self.layer_coarse = modules[layer_coarse_name]

        self.activations = {}
        self.gradients   = {}

        self.layer_fine.register_forward_hook(self._activation_hook('fine'))
        self.layer_fine.register_full_backward_hook(self._gradient_hook('fine'))
        self.layer_coarse.register_forward_hook(self._activation_hook('coarse'))
        self.layer_coarse.register_full_backward_hook(self._gradient_hook('coarse'))

    def _activation_hook(self, name):
        def hook(module, inp, out):
            self.activations[name] = out.detach().cpu()
        return hook

    def _gradient_hook(self, name):
        def hook(module, grad_in, grad_out):
            self.gradients[name] = grad_out[0].detach().cpu()
        return hook

    def _gradcampp_single(self, act_tensor, grad_tensor, out_size):
        """Tính GradCAM++ map từ activations + gradients của 1 layer."""
        act  = act_tensor[0].numpy()   # [C, H, W]
        grad = grad_tensor[0].numpy()  # [C, H, W]

        eps   = 1e-7
        grad2 = grad ** 2
        grad3 = grad ** 3
        # sum activations per channel  → [C, 1, 1]
        sum_act = np.sum(act, axis=(1, 2), keepdims=True)
        denom   = 2.0 * grad2 + sum_act * grad3 + eps
        alpha   = grad2 / denom                                     # [C, H, W]
        weights = np.sum(alpha * np.maximum(grad, 0), axis=(1, 2))  # [C]

        cam = np.sum(weights[:, None, None] * act, axis=0)  # [H, W]
        cam = np.maximum(cam, 0)

        cam = cv2.resize(cam, out_size)  # (W, H)
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def generate(self, input_tensor, target_class):
        input_tensor = input_tensor.to(self.device)
        self.activations.clear()
        self.gradients.clear()

        self.model.zero_grad()
        output     = self.model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        confidence = torch.softmax(output, dim=1)[0, target_class].item()

        output[0, target_class].backward()

        W, H = input_tensor.shape[3], input_tensor.shape[2]

        cam_fine   = self._gradcampp_single(self.activations['fine'],   self.gradients['fine'],   (W, H))
        cam_coarse = self._gradcampp_single(self.activations['coarse'], self.gradients['coarse'], (W, H))

        fused = self.weight_fine * cam_fine + self.weight_coarse * cam_coarse
        if fused.max() > fused.min():
            fused = (fused - fused.min()) / (fused.max() - fused.min())

        return fused, confidence, pred_class

In [8]:
gradcam_v2 = MultiScaleGradCAMpp(
    model=model,
    layer_fine_name=layer6_name,
    layer_coarse_name=layer8_name,
    device=device,
    weight_fine=0.6,
)

infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('MultiScale GradCAM++ ready')
print(f'  fine layer  : {layer6_name}')
print(f'  coarse layer: {layer8_name}')
print(f'  fusion      : 0.6 × fine + 0.4 × coarse')

MultiScale GradCAM++ ready
  fine layer  : base.features.6.3.block.3.0
  coarse layer: base.features.8.0
  fusion      : 0.6 × fine + 0.4 × coarse


## Improved Mask Pipeline

Ba cải tiến so với v1:
1. **Otsu threshold** (per-image adaptive) thay fixed=0.5: tự tìm threshold tốt nhất cho distribution heatmap của từng ảnh cụ thể
2. **Ellipse kernel** thay rect kernel: phù hợp hơn với hình dạng tròn/oval của đốm bệnh
3. **Connected component filter** (min 50px): loại bỏ noise nhỏ, giữ các vùng bệnh có diện tích thực sự

In [9]:
def adaptive_threshold_otsu(cam_map):
    """Otsu per-image — tốt hơn fixed 0.5 vì tự điều chỉnh theo distribution."""
    cam_u8 = (cam_map * 255).astype(np.uint8)
    _, mask = cv2.threshold(cam_u8, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask.astype(np.uint8)


def filter_connected_components(mask, min_area=50):
    """Giữ lại connected components có diện tích >= min_area pixel."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    filtered = np.zeros_like(mask)
    for i in range(1, num_labels):  # 0 là background
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            filtered[labels == i] = 1
    return filtered


def create_pseudo_mask_v2(heatmap, min_cc_area=50):
    """Pipeline: Otsu → Morphological (ellipse kernel) → CC filter."""
    # 1) Adaptive threshold
    mask = adaptive_threshold_otsu(heatmap)

    # 2) Morphological: ellipse kernel tốt hơn rect cho vùng tròn như đốm lá
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # lấp lỗ hổng
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)  # xóa noise nhỏ

    # 3) CC filtering
    mask = filter_connected_components(mask, min_area=min_cc_area)
    return mask.astype(np.uint8)


print('Helper functions ready')

Helper functions ready


## Generate Pseudo-Label v2

Healthy Leaf không có vùng tổn thương thực — mask của lớp này trong v1 là noise (coverage 26.4% khuếch tán). Trong v2: xử lý Healthy Leaf bình thường với pipeline, nhưng trong notebook 08 sẽ set all-zero mask cho Healthy Leaf khi training segmentation.

Giới hạn `MAX_VIZ_PER_CLASS=10` để tránh tốn RAM — chỉ cần đủ mẫu để visualize, không cần lưu hết 3K ảnh vào memory.

In [ ]:
pseudo_dir_v2    = Path(os.path.join(proj_root, 'notebooks', 'data', 'pseudo_labels_v2_train'))
processed_dir_v2 = Path(os.path.join(proj_root, 'notebooks', 'data', 'processed_v2_train'))
pseudo_dir_v2.mkdir(parents=True, exist_ok=True)
processed_dir_v2.mkdir(parents=True, exist_ok=True)

stats_v2            = []
results_by_class_v2 = defaultdict(list)
MAX_VIZ_PER_CLASS   = 10  # giới hạn RAM cho visualization

for img_path, cls_idx, cls_name in tqdm(selected_samples, desc="GradCAM++ v2"):
    try:
        orig_img = Image.open(img_path).convert("RGB").resize((224, 224))
        orig_arr = np.array(orig_img)

        input_tensor = infer_transform(orig_img).unsqueeze(0)

        heatmap, confidence, pred_class = gradcam_v2.generate(input_tensor, cls_idx)

        mask = create_pseudo_mask_v2(heatmap, min_cc_area=50)

        fname     = Path(img_path).name
        base_name = f"{cls_name}_{fname}"

        # Lưu mask
        cv2.imwrite(str(pseudo_dir_v2 / f"{base_name}_pseudo.png"),
                    (mask * 255).astype(np.uint8))
        # Lưu ảnh gốc
        orig_img.save(str(processed_dir_v2 / base_name))

        stats_v2.append({
            "filename":      base_name,
            "original_path": img_path,
            "class_name":    cls_name,
            "class_idx":     cls_idx,
            "pred_class":    pred_class,
            "confidence":    float(confidence),
            "mask_coverage": float(mask.mean()),
        })

        # Lưu data để viz (chỉ giữ MAX_VIZ_PER_CLASS đầu mỗi lớp)
        if len(results_by_class_v2[cls_name]) < MAX_VIZ_PER_CLASS:
            heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heatmap_rgb     = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
            gradcam_overlay = cv2.addWeighted(orig_arr, 0.6, heatmap_rgb, 0.4, 0)
            mask_rgb        = np.zeros_like(orig_arr)
            mask_rgb[:, :, 1] = mask * 255
            pseudo_overlay  = cv2.addWeighted(orig_arr, 0.7, mask_rgb, 0.3, 0)
            results_by_class_v2[cls_name].append({
                "orig_arr":        orig_arr,
                "heatmap":         heatmap,
                "gradcam_overlay": gradcam_overlay,
                "mask":            mask,
                "pseudo_overlay":  pseudo_overlay,
                "confidence":      float(confidence),
                "pred_class":      pred_class,
                "true_class":      cls_idx,
            })

    except Exception as e:
        print(f"  Lỗi {img_path}: {e}")

with open(pseudo_dir_v2 / "pseudo_label_stats_v2.json", "w") as f:
    json.dump(stats_v2, f, indent=2)

print(f"\nGenerated v2 pseudo-labels: {len(stats_v2)}")
print(f"Masks → {pseudo_dir_v2}")
print(f"Images → {processed_dir_v2}")

## Visualization

Hai chart:
1. **Best sample per class** — ảnh confidence cao nhất, xem GradCAM++ mask có sắc nét hơn v1 không
2. **V1 vs V2 comparison** — so sánh mask cùng một ảnh để thấy hiệu quả cải tiến

In [ ]:
n_classes = len(CLASS_NAMES)
fig, axes = plt.subplots(n_classes, 3, figsize=(15, 5 * n_classes))
fig.suptitle("Pseudo-Label v2 — GradCAM++ Multi-Scale + Otsu + CC Filter", fontsize=13, fontweight="bold")

for i, (cls_name, cls_label, cls_color) in enumerate(zip(CLASS_NAMES, CLASS_LABELS, CLASS_COLORS)):
    items = results_by_class_v2.get(cls_name, [])
    if not items:
        continue
    best = max(items, key=lambda x: x["confidence"])

    axes[i, 0].imshow(best["orig_arr"])
    axes[i, 0].set_title(f"{cls_label}\nOriginal", fontsize=10, fontweight="bold", color=cls_color)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(best["gradcam_overlay"])
    axes[i, 1].set_title(f"GradCAM++ Overlay\nConf={best['confidence']:.3f}", fontsize=10)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(best["pseudo_overlay"])
    axes[i, 2].set_title(f"Pseudo Mask v2\nCoverage={best['mask'].mean():.3f}", fontsize=10)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "pseudo_label_v2_per_class.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {VIZ_DIR}/pseudo_label_v2_per_class.png")

In [ ]:
demo_cls   = CLASS_NAMES[0]  # ALGAL_LEAF_SPOT — lớp có nhiều đốm nhỏ nhất
demo_items = results_by_class_v2.get(demo_cls, [])

if demo_items:
    demo    = demo_items[0]
    orig    = demo["orig_arr"]
    heatmap = demo["heatmap"]

    # V1: fixed threshold 0.5, rect kernel
    mask_v1 = (heatmap >= 0.5).astype(np.uint8)
    kernel  = np.ones((5, 5), np.uint8)
    mask_v1 = cv2.morphologyEx(mask_v1, cv2.MORPH_CLOSE, kernel)
    mask_v1 = cv2.morphologyEx(mask_v1, cv2.MORPH_OPEN,  kernel)

    # V2: Otsu + CC filter
    mask_v2 = demo["mask"]

    def overlay_mask(img, mask, color_ch=1):
        rgb = np.zeros_like(img)
        rgb[:, :, color_ch] = mask * 255
        return cv2.addWeighted(img, 0.7, rgb, 0.3, 0)

    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle(f"V1 vs V2 Mask Comparison — {CLASS_LABELS[0]}", fontsize=13, fontweight="bold")

    axes2[0].imshow(orig)
    axes2[0].set_title("Original", fontsize=10); axes2[0].axis("off")

    axes2[1].imshow(heatmap, cmap="jet")
    axes2[1].set_title("GradCAM++ Heatmap\n(multi-scale)", fontsize=10); axes2[1].axis("off")

    axes2[2].imshow(overlay_mask(orig, mask_v1, color_ch=0))
    axes2[2].set_title("V1 Mask\n(fixed thresh=0.5, rect kernel)", fontsize=10); axes2[2].axis("off")

    axes2[3].imshow(overlay_mask(orig, mask_v2, color_ch=1))
    axes2[3].set_title("V2 Mask\n(Otsu + ellipse + CC filter)", fontsize=10); axes2[3].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "v1_vs_v2_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {VIZ_DIR}/v1_vs_v2_comparison.png")

## Đánh Giá Chất Lượng — So Sánh v1 vs v2

So sánh coverage distribution và accuracy giữa v1 (notebook 04) và v2. Coverage v2 kỳ vọng cao hơn v1 vì Otsu threshold thường chọn ngưỡng thấp hơn 0.5 trên heatmap bimodal — nhưng CC filter loại bỏ các vùng nhỏ rải rác, giữ chỉ vùng lớn thực sự.

In [ ]:
df_v2 = pd.DataFrame(stats_v2)

v1_stats_path = os.path.join(proj_root, 'notebooks', 'data', 'pseudo_labels_train', 'pseudo_label_stats.json')
if os.path.exists(v1_stats_path):
    with open(v1_stats_path) as f:
        df_v1 = pd.DataFrame(json.load(f))

    print("=" * 55)
    print(f'{"Metric":<25} {"V1 (nb04)":>12} {"V2 (nb07)":>12}')
    print("-" * 55)
    print(f'{"Total samples":<25} {len(df_v1):>12} {len(df_v2):>12}')
    print(f'{"Accuracy (pred==true)":<25} {(df_v1["pred_class"]==df_v1["class_idx"]).mean():>12.2%} '
          f'{(df_v2["pred_class"]==df_v2["class_idx"]).mean():>12.2%}')
    print(f'{"Mean confidence":<25} {df_v1["confidence"].mean():>12.4f} {df_v2["confidence"].mean():>12.4f}')
    print(f'{"Mean mask coverage":<25} {df_v1["mask_coverage"].mean():>12.4f} {df_v2["mask_coverage"].mean():>12.4f}')
    print(f'{"Near-empty masks (<0.05)":<25} {(df_v1["mask_coverage"]<0.05).sum():>12} '
          f'{(df_v2["mask_coverage"]<0.05).sum():>12}')
    print(f'{"Over-large masks (>0.6)":<25} {(df_v1["mask_coverage"]>0.6).sum():>12} '
          f'{(df_v2["mask_coverage"]>0.6).sum():>12}')
    print("=" * 55)
else:
    print("V1 stats không tìm thấy. Hiển thị V2 stats:")

print("\n=== V2 Per-Class Stats ===")
print(df_v2.groupby("class_name")[["confidence", "mask_coverage"]].agg(["mean", "std"]).round(3))

In [ ]:
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Mask Coverage Analysis — V1 vs V2", fontsize=13, fontweight="bold")

if os.path.exists(v1_stats_path):
    axes3[0].hist(df_v1["mask_coverage"], bins=30, alpha=0.7, label="V1 (fixed 0.5)", color="#D55E00", edgecolor="white")
    axes3[0].hist(df_v2["mask_coverage"], bins=30, alpha=0.7, label="V2 (Otsu)", color="#009E73", edgecolor="white")
else:
    axes3[0].hist(df_v2["mask_coverage"], bins=30, color="#009E73", edgecolor="white")
axes3[0].set_title("Coverage Distribution", fontsize=12, fontweight="bold")
axes3[0].set_xlabel("Coverage (fraction of image)")
axes3[0].set_ylabel("Count")
axes3[0].legend()
axes3[0].grid(True, alpha=0.3)

per_cls = df_v2.groupby("class_name")["mask_coverage"].mean()
labels  = [IDX_TO_LABEL[CLASS_TO_IDX[n]] for n in per_cls.index]
axes3[1].bar(labels, per_cls.values, color=CLASS_COLORS, alpha=0.88, edgecolor="white", linewidth=1.5)
axes3[1].set_title("V2 Mean Coverage per Class", fontsize=12, fontweight="bold")
axes3[1].set_xlabel("Class")
axes3[1].set_ylabel("Mean Coverage")
axes3[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
axes3[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "v2_coverage_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {VIZ_DIR}/v2_coverage_analysis.png")

---

## Kết luận

Pseudo-label v2 cải thiện đáng kể chất lượng mask so với v1 thông qua 5 kỹ thuật: GradCAM++ multi-scale, Otsu threshold, ellipse kernel, CC filter. Mask có biên sắc nét hơn và phân biệt được các đốm rời rạc.

**Lưu ý cho notebook 08:** Healthy Leaf coverage vẫn cao (~40%) do không có lesion thực. Khi training segmentation (NB08), nên set all-zero mask cho Healthy Leaf hoặc loại khỏi training để tránh false positive label.

**Bước tiếp theo:** Notebook 08 dùng pseudo-label v2 này để train UNet++ với Albumentations augmentation mạnh và FocalDice loss — kỳ vọng cải thiện IoU/Dice so với baseline v1 ở notebook 05.